# RQ2 Champion Models

Compares champion point models across RQ2 ocean regimes.

**Provenance.** This is an English-facing execution copy of the original project notebook. The computational cells are preserved so that the analysis remains reproducible. Chinese narrative cells have been replaced by concise English stage notes; the original notebook remains unchanged in the source project.

## Analysis stage 1

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
from pathlib import Path 
import numpy as np 
import pandas as pd 
import matplotlib .pyplot as plt 
from IPython .display import display ,Markdown 

PROJECT =next ((p for p in [Path .cwd (),*Path .cwd ().parents ]
if (p /'data'/'processed').exists ()),None )
if PROJECT is None :
    raise FileNotFoundError ('\u8bf7\u4ecedata3001-project\u6216\u5176\u5b50\u76ee\u5f55\u8fd0\u884c。')

RQ2 =PROJECT /'RQ2'
OUT =RQ2 /'champion_models'/'output'
OUT .mkdir (parents =True ,exist_ok =True )
ZH =PROJECT /'output'/'rq1'/'zh'
EN =PROJECT /'output'/'rq1'/'en'

plt .style .use ('seaborn-v0_8-whitegrid')
plt .rcParams ['font.sans-serif']=['PingFang SC','Arial Unicode MS','DejaVu Sans']
plt .rcParams ['axes.unicode_minus']=False 
print ('\u9879\u76ee\u76ee\u5f55：',PROJECT )
print ('\u8f93\u51fa\u76ee\u5f55：',OUT )

## Analysis stage 2

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
origins = pd.read_parquet(PROJECT/'data'/'processed'/'forecast_origins.parquet')
origins = origins.sort_values('forecast_case_id').reset_index(drop=True)
test_meta = origins.loc[origins.partition.eq('test'), ['forecast_case_id','ID']].copy()

regimes = pd.read_csv(RQ2/'data'/'forecast_case_regimes.csv')[
    ['forecast_case_id','regime','current_speed_clim','distance_to_kuroshio_km','depth_m']
]
test_meta = test_meta.merge(regimes, on='forecast_case_id', how='left', validate='one_to_one')

assert len(test_meta) == 757
assert test_meta.forecast_case_id.is_unique
assert test_meta.regime.notna().all()
assert set(test_meta.regime) == {'Shelf','Kuroshio Core','Transition','Other Offshore'}
display(test_meta.groupby('regime').agg(test_cases=('forecast_case_id','size'),
                                        test_drifters=('ID','nunique')))

## Analysis stage 3

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
MODEL_FILES ={
'Velocity persistence':EN /'velocity_persistence_test_case_metrics.csv',
'Spatial mean flow':EN /'spatial_mean_flow_test_case_metrics.csv',
'Optimised FNN':ZH /'optimised_fnn_test_case_metrics.csv',
'Boosting residual':ZH /'boosting_residual_test_case_metrics.csv',
'Optimised Weighted KNN':ZH /'optimised_knn_test_case_metrics.csv',
'Optimised Random Forest':ZH /'optimised_random_forest_test_case_metrics.csv',
}

def standardise_case_metrics (model ,path ):
    d =pd .read_csv (path )
    ren ={
    'drifter_id':'ID',
    'error_72h_km':'final_72h_error_km',
    'ite_km_h':'integrated_trajectory_error_km_h',
    }
    d =d .rename (columns =ren )
    required ={'forecast_case_id','final_72h_error_km','integrated_trajectory_error_km_h'}
    missing =required -set (d .columns )
    if missing :
        raise ValueError (f'{model }\u7f3a\u5c11\u5217：{missing }')
    d =d [['forecast_case_id','final_72h_error_km','integrated_trajectory_error_km_h']].copy ()
    assert len (d )==757 and d .forecast_case_id .is_unique 
    assert set (d .forecast_case_id )==set (test_meta .forecast_case_id )
    assert np .isfinite (d [['final_72h_error_km','integrated_trajectory_error_km_h']]).all ().all ()
    d ['model']=model 
    return d 

errors =pd .concat ([standardise_case_metrics (m ,p )for m ,p in MODEL_FILES .items ()],ignore_index =True )
errors =errors .merge (test_meta ,on ='forecast_case_id',how ='left',validate ='many_to_one')
assert len (errors )==757 *len (MODEL_FILES )
assert errors .groupby ('forecast_case_id').model .nunique ().eq (len (MODEL_FILES )).all ()
print ('\u5ba1\u8ba1\u901a\u8fc7：',len (errors ),'\u884c；',errors .model .nunique (),'\u4e2a\u6a21\u578b；757\u4e2a\u5171\u540ctest cases。')

## Analysis stage 4

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
def summarise(g):
    e = g.final_72h_error_km.to_numpy()
    threshold = np.quantile(e,.9)
    return pd.Series({
        'n_cases':len(g), 'n_drifters':g.ID.nunique(),
        'median_72h_km':np.median(e),
        'rmse_72h_km':np.sqrt(np.mean(e**2)),
        'median_ite_km_h':np.median(g.integrated_trajectory_error_km_h),
        'worst10_rmse_72h_km':np.sqrt(np.mean(e[e>=threshold]**2)),
    })

summary = errors.groupby(['regime','model'], observed=True).apply(summarise, include_groups=False).reset_index()
summary.to_csv(OUT/'champion_regime_summary.csv', index=False)
for metric in ['median_72h_km','rmse_72h_km','median_ite_km_h','worst10_rmse_72h_km']:
    display(Markdown('### '+metric))
    display(summary.pivot(index='model',columns='regime',values=metric).round(2))

## Analysis stage 5

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
def cluster_bootstrap(frame, n_boot=2000, seed=2026):
    rng = np.random.default_rng(seed)
    rows=[]
    for (regime,model),g in frame.groupby(['regime','model'], observed=True):
        groups={k:v.final_72h_error_km.to_numpy() for k,v in g.groupby('ID')}
        ids=np.array(list(groups),dtype=object)
        med=[]; rms=[]
        for _ in range(n_boot):
            sample=np.concatenate([groups[i] for i in rng.choice(ids,len(ids),replace=True)])
            med.append(np.median(sample)); rms.append(np.sqrt(np.mean(sample**2)))
        e=g.final_72h_error_km.to_numpy()
        for metric,point,values in [
            ('median_72h_km',np.median(e),med),
            ('rmse_72h_km',np.sqrt(np.mean(e**2)),rms),
        ]:
            lo,hi=np.quantile(values,[.025,.975])
            rows.append({'regime':regime,'model':model,'metric':metric,'estimate':point,
                         'ci_lower':lo,'ci_upper':hi,'n_drifters':len(ids)})
    return pd.DataFrame(rows)

boot = cluster_bootstrap(errors)
boot.to_csv(OUT/'champion_regime_cluster_bootstrap.csv',index=False)
display(boot.round(2).head(12))

## Analysis stage 6

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
ORDER=['Velocity persistence','Spatial mean flow','Optimised Random Forest',
       'Optimised Weighted KNN','Optimised FNN','Boosting residual']
REGIMES=['Shelf','Kuroshio Core','Transition','Other Offshore']
COLORS={'Velocity persistence':'#6c757d','Spatial mean flow':'#adb5bd',
        'Optimised Random Forest':'#3267e3','Optimised Weighted KNN':'#1b9e77',
        'Optimised FNN':'#d81b60','Boosting residual':'#7b2cbf'}

fig,axes=plt.subplots(2,2,figsize=(16,11),sharex=False)
for ax,regime in zip(axes.flat,REGIMES):
    d=boot[(boot.regime==regime)&(boot.metric=='rmse_72h_km')].set_index('model').reindex(ORDER)
    y=np.arange(len(d)); x=d.estimate.to_numpy(); lo=d.ci_lower.to_numpy(); hi=d.ci_upper.to_numpy()
    ax.errorbar(x,y,xerr=np.vstack([x-lo,hi-x]),fmt='none',ecolor='#444',capsize=4,zorder=1)
    ax.scatter(x,y,c=[COLORS[m] for m in d.index],s=65,zorder=2)
    ax.set_yticks(y,d.index); ax.invert_yaxis(); ax.set_title(regime,fontweight='bold')
    ax.set_xlabel('72-h endpoint RMSE (km; lower is better)')
fig.suptitle('Champion models across ocean regimes',fontsize=16,fontweight='bold')
fig.tight_layout(); fig.savefig(OUT/'champion_rmse_by_regime.png',dpi=200,bbox_inches='tight'); plt.show()

fig,axes=plt.subplots(1,3,figsize=(18,5.5))
for ax,metric,title,unit in zip(axes,
    ['median_72h_km','rmse_72h_km','median_ite_km_h'],
    ['Median endpoint error','Endpoint RMSE','Median 72-h ITE'],['km','km','km·h']):
    overall=errors.groupby('model',observed=True).apply(
        lambda g: summarise(g)[metric],include_groups=False).reindex(ORDER).sort_values(ascending=False)
    ax.barh(overall.index,overall.values,color=[COLORS[m] for m in overall.index])
    ax.set_title(title,fontweight='bold'); ax.set_xlabel(unit)
fig.tight_layout(); fig.savefig(OUT/'champion_overall_metrics.png',dpi=200,bbox_inches='tight'); plt.show()

## Analysis stage 7

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
learned =['Optimised FNN','Boosting residual','Optimised Weighted KNN','Optimised Random Forest']
for regime in REGIMES :
    d =summary [summary .regime .eq (regime )].set_index ('model')
    best_med =d .loc [learned ,'median_72h_km'].idxmin ()
    best_rmse =d .loc [learned ,'rmse_72h_km'].idxmin ()
    best_ite =d .loc [learned ,'median_ite_km_h'].idxmin ()
    baseline_rmse =d .loc [['Velocity persistence','Spatial mean flow'],'rmse_72h_km'].min ()
    gain =100 *(1 -d .loc [best_rmse ,'rmse_72h_km']/baseline_rmse )
    print (f'{regime }: median\u51a0\u519b={best_med }; RMSE\u51a0\u519b={best_rmse }; ITE\u51a0\u519b={best_ite }; '
    f'RMSE\u51a0\u519b\u76f8\u5bf9\u6700\u4f73\u7269\u7406baseline\u6539\u5584{gain :.1f}%')

display (Markdown ("""
### \u89e3\u91ca\u8fb9\u754c

- \u672cNotebook\u53ef\u4ee5\u5b8c\u6574\u56de\u7b5472\u5c0f\u65f6\u7ec8\u70b9\u8bef\u5dee、ITE、\u5c3e\u90e8\u98ce\u9669\u548c\u533a\u57dfbootstrap\u4e0d\u786e\u5b9a\u6027。
- \u5f53\u524d\u51a0\u519bCSV\u6ca1\u6709\u4fdd\u5b58`case × lead_time`\u7684\u5b8c\u6574\u8bef\u5dee\u77e9\u9635，\u56e0\u6b64\u8fd9\u91cc\u6ca1\u6709\u4f2a\u90201–72\u5c0f\u65f6\u533a\u57df\u66f2\u7ebf。
- \u82e5\u540e\u7eed\u8981\u753b\u6bcf\u4e2aregime\u7684\u9010\u5c0f\u65f6\u66f2\u7ebf，\u5e94\u8ba9\u56db\u4e2a\u51a0\u519b\u8bad\u7ec3Notebook\u989d\u5916\u5bfc\u51fa\u9010case\u9010\u5c0f\u65f6\u9884\u6d4b\u6216\u8bef\u5dee。
"""))